# Diseño de Muro de Concreto - Flexocompresión (NTC 2023)
Este programa calcula la profundidad del eje neutro ($c$) y el área de acero ($A_s$) requerida para la sección transversal especificada en el archivo `MC-01.jpeg`.

**Consideraciones:**
- $f'c = 250 \text{ kg/cm}^2$
- $fy = 4200 \text{ kg/cm}^2$
- Norma: NTC-Concreto 2023 (CDMX)
- Altura del muro = 4 m

In [1]:
import numpy as np
from scipy.optimize import fsolve

# 1. DATOS DE ENTRADA
# Materiales
fc = 250      # kg/cm2
fy = 4200     # kg/cm2
Es = 2e6      # kg/cm2 (Módulo de elasticidad del acero)

# Cargas Últimas (convertidas a kg y cm)
Pu = 1694 * 1000          # kg
Mu = 3933.96 * 1000 * 100 # kg-cm
Vu = 1187.75 * 1000       # kg

# Geometría (según MC-01.jpeg)
b_alma = 50   # cm
h_alma = 428  # cm
b_patin = 90  # cm
h_patin = 150 # cm
L_total = h_alma + h_patin # 578 cm

# Parámetros del bloque equivalente (NTC 2023 para f'c <= 280)
f_bic = 0.85 * fc
beta1 = 0.85

print("Datos inicializados correctamente.")

Datos inicializados correctamente.


## Caso 1: Compresión en el elemento de 90x150 cm (El Patín)
En este caso, la fibra más comprimida está en el borde del patín de 90 cm de ancho.

In [2]:
def equilibrio_caso1(vars):
    c, As = vars
    a = beta1 * c
    
    # Fuerza de compresión en el concreto (Cc)
    if a <= h_patin:
        Cc = f_bic * b_patin * a
        # Centroide de la compresión desde la fibra extrema
        yc = a / 2
    else:
        Cc1 = f_bic * b_patin * h_patin
        Cc2 = f_bic * b_alma * (a - h_patin)
        Cc = Cc1 + Cc2
        yc = (Cc1 * (h_patin/2) + Cc2 * (h_patin + (a-h_patin)/2)) / Cc
        
    # Suponiendo acero concentrado en los extremos para el equilibrio simplificado inicial
    # (El ingeniero puede ajustar la distribución de varillas posteriormente)
    # d = peralte efectivo estimado
    d = L_total - 10 # recubrimiento supuesto de 10 cm al centroide del acero a tensión
    
    # Esfuerzo en el acero
    es = 0.003 * (d - c) / c
    fs = min(Es * es, fy)
    
    # Ecuaciones de equilibrio
    # 1. Suma de fuerzas: Pu = Cc - As*fs
    eq1 = Pu - (Cc - As * fs)
    
    # 2. Suma de momentos respecto al centroide plástico (aprox L_total/2)
    # Centro geométrico simplificado (para momento):
    cg = L_total / 2
    eq2 = Mu - (Cc * (cg - yc) + As * fs * (d - cg))
    
    return [eq1, eq2]

# Valores semilla iniciales: c = 100 cm, As = 100 cm2
res_caso1 = fsolve(equilibrio_caso1, [150, 200])
c_1, As_1 = res_caso1
print(f"--- CASO 1: COMPRESIÓN EN EL PATÍN (90x150) ---")
print(f"Profundidad del Eje Neutro (c): {c_1:.2f} cm")
print(f"Área de acero a tensión requerida (As): {As_1:.2f} cm²")

--- CASO 1: COMPRESIÓN EN EL PATÍN (90x150) ---
Profundidad del Eje Neutro (c): 101.50 cm
Área de acero a tensión requerida (As): -10.48 cm²


## Caso 2: Compresión en el extremo del alma (50 cm de ancho)
Aquí la fibra más comprimida está en el extremo de 50 cm.

In [3]:
def equilibrio_caso2(vars):
    c, As = vars
    a = beta1 * c
    
    # Fuerza de compresión en el concreto (Cc)
    if a <= h_alma:
        Cc = f_bic * b_alma * a
        yc = a / 2
    else:
        Cc1 = f_bic * b_alma * h_alma
        Cc2 = f_bic * b_patin * (a - h_alma)
        Cc = Cc1 + Cc2
        yc = (Cc1 * (h_alma/2) + Cc2 * (h_alma + (a-h_alma)/2)) / Cc
        
    d = L_total - 10 # recubrimiento supuesto
    
    es = 0.003 * (d - c) / c
    fs = min(Es * es, fy)
    
    # Ecuaciones de equilibrio
    eq1 = Pu - (Cc - As * fs)
    
    cg = L_total / 2
    eq2 = Mu - (Cc * (cg - yc) + As * fs * (d - cg))
    
    return [eq1, eq2]

res_caso2 = fsolve(equilibrio_caso2, [150, 200])
c_2, As_2 = res_caso2
print(f"--- CASO 2: COMPRESIÓN EN EL ALMA (50 cm) ---")
print(f"Profundidad del Eje Neutro (c): {c_2:.2f} cm")
print(f"Área de acero a tensión requerida (As): {As_2:.2f} cm²")

--- CASO 2: COMPRESIÓN EN EL ALMA (50 cm) ---
Profundidad del Eje Neutro (c): 198.22 cm
Área de acero a tensión requerida (As): 22.91 cm²
